In [ ]:
import sys
!{sys.executable} -m pip install lightgbm scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv(r"C:\Users\yajko\Downloads\LendingClub_Cleaned_Data.csv", low_memory=False)

In [ ]:
df = df[[
    'int_rate',
    'term',
    'Avg_Fico',
    'dti',
    'loan_amnt',
    'inq_last_6mths',
    'bc_util',
    'revol_util',
    'total_rev_hi_lim',
    'mths_since_recent_inq',
    'verification_status',
    'home_ownership',
    'purpose',
    'application_type',
    'earliest_cr_line',
    'issue_d']].copy()

In [ ]:
df['int_rate'] = pd.to_numeric(
    df['int_rate'].astype(str).str.replace('%', '', regex=False).str.strip(),
    errors='coerce')

df['term'] = df['term'].astype(str).str.replace(" months", "", regex=False)
df['term'] = df['term'].str.replace(" ", "", regex=False)
df['term'] = pd.to_numeric(df['term'], errors='coerce')

df['verification_status'] = df['verification_status'].str.replace("Source Verified", "Verified")
df['verification_status'] = df['verification_status'].str.replace("Not Verified", "Not_Verified")

In [ ]:
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%Y-%m-%d', errors='coerce')
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
df['credit_history_yrs'] = (df['issue_d'] - df['earliest_cr_line']).dt.days / 365.25

df = df.drop(columns=['issue_d', 'earliest_cr_line'])

In [ ]:
df = df[df['int_rate'].notna()].copy()

In [ ]:
columns_to_encode = ['verification_status',
                     'home_ownership', 'purpose', 'application_type']
df_dummies = pd.get_dummies(
    df,
    columns=columns_to_encode,
    drop_first=True,
    dtype=int
)

In [ ]:
df_dummies.head()

In [ ]:
y = df_dummies['int_rate']
X = df_dummies.drop(columns=['int_rate'])
print("Rows:", len(X), " Features:", X.shape[1])

In [ ]:
params = dict(objective="regression", n_estimators=600, learning_rate=0.05,
              num_leaves=64, subsample=0.8, colsample_bytree=0.8,
              random_state=42, n_jobs=-1, verbose=-1)

kf = KFold(n_splits=10, shuffle=True, random_state=42)
mae_list, rmse_list, r2_list = [], [], []
oof_pred = np.zeros(len(y))              
importances = np.zeros(X.shape[1])

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

   
    X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

    model = lgb.LGBMRegressor(**params)
    model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], eval_metric="rmse",
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])

    pred = model.predict(X_test)
    oof_pred[test_idx] = pred

    mae_list.append(mean_absolute_error(y_test, pred))
    rmse_list.append(np.sqrt(mean_squared_error(y_test, pred)))
    r2_list.append(r2_score(y_test, pred))
    importances += model.booster_.feature_importance(importance_type="gain")

    print(f"Fold {fold:2d}: MAE={mae_list[-1]:.3f}  RMSE={rmse_list[-1]:.3f}  R2={r2_list[-1]:.3f}")

print("\n=== 10-fold CV (mean +/- std) ===")
print(f"MAE : {np.mean(mae_list):.3f} +/- {np.std(mae_list):.3f}")
print(f"RMSE: {np.mean(rmse_list):.3f} +/- {np.std(rmse_list):.3f}")
print(f"R2  : {np.mean(r2_list):.3f} +/- {np.std(r2_list):.3f}")

importances = importances / kf.get_n_splits()

In [ ]:
imp = pd.Series(importances, index=X.columns).sort_values()
top = imp.tail(20)
plt.figure(figsize=(8, 7))
plt.barh(top.index, top.values)
plt.xlabel("Average gain importance")
plt.title("LightGBM Feature Importance for Interest Rate (no sub_grade, top 20)")
plt.tight_layout()
plt.show()

In [ ]:
samp = np.random.RandomState(42).choice(len(y), size=min(20000, len(y)), replace=False)
plt.figure(figsize=(6, 6))
plt.scatter(y.iloc[samp], oof_pred[samp], s=4, alpha=0.2)
plt.plot([y.min(), y.max()], [y.min(), y.max()], color="red", linewidth=1)
plt.xlabel("Actual interest rate (%)")
plt.ylabel("Predicted interest rate (%)")
plt.title("Actual vs Predicted Interest Rate (no sub_grade)")
plt.tight_layout()
plt.show()

In [ ]:
resid = y.values - oof_pred
plt.figure(figsize=(7, 5))
plt.hist(resid, bins=60)
plt.xlabel("Residual (actual minus predicted, %)")
plt.ylabel("Count")
plt.title("Distribution of Residuals (no sub_grade)")
plt.tight_layout()
plt.show()

In [ ]:
import os

out_dir = "lightgbm_no_subgrade_outputs"
os.makedirs(out_dir, exist_ok=True)

# 1) Write the metrics to a text file
with open(os.path.join(out_dir, "rq2_lightgbm_no_subgrade_results.txt"), "w") as f:
    f.write("LightGBM Regression - RQ2 Interest Rate Prediction (NO sub_grade)\n")
    f.write("=" * 50 + "\n")
    f.write(f"Rows used: {len(y):,}\n")
    f.write(f"Features:  {X.shape[1]}\n\n")
    f.write("Per-fold results:\n")
    for i, (mae, rmse, r2) in enumerate(zip(mae_list, rmse_list, r2_list), 1):
        f.write(f"  Fold {i:2d}: MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.3f}\n")
    f.write("\n10-fold CV (mean +/- std):\n")
    f.write(f"  MAE : {np.mean(mae_list):.3f} +/- {np.std(mae_list):.3f}\n")
    f.write(f"  RMSE: {np.mean(rmse_list):.3f} +/- {np.std(rmse_list):.3f}\n")
    f.write(f"  R2  : {np.mean(r2_list):.3f} +/- {np.std(r2_list):.3f}\n")

imp = pd.Series(importances, index=X.columns).sort_values().tail(20)
plt.figure(figsize=(8, 7))
plt.barh(imp.index, imp.values)
plt.xlabel("Average gain importance")
plt.title("LightGBM Feature Importance for Interest Rate (no sub_grade, top 20)")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "rq2_feature_importance.png"), dpi=150)
plt.close()

samp = np.random.RandomState(42).choice(len(y), size=min(20000, len(y)), replace=False)
plt.figure(figsize=(6, 6))
plt.scatter(y.iloc[samp], oof_pred[samp], s=4, alpha=0.2)
plt.plot([y.min(), y.max()], [y.min(), y.max()], color="red", linewidth=1)
plt.xlabel("Actual interest rate (%)")
plt.ylabel("Predicted interest rate (%)")
plt.title("Actual vs Predicted Interest Rate (no sub_grade)")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "rq2_actual_vs_predicted.png"), dpi=150)
plt.close()

resid = y.values - oof_pred
plt.figure(figsize=(7, 5))
plt.hist(resid, bins=60)
plt.xlabel("Residual (actual minus predicted, %)")
plt.ylabel("Count")
plt.title("Distribution of Residuals (no sub_grade)")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "rq2_residuals.png"), dpi=150)
plt.close()

print("Saved to folder:", os.path.abspath(out_dir))
for fn in ["rq2_lightgbm_no_subgrade_results.txt", "rq2_feature_importance.png",
           "rq2_actual_vs_predicted.png", "rq2_residuals.png"]:
    print("  ", fn)